# March Madness - Training Pipeline

Run this notebook once per season to calibrate `ModelParams` from historical NCAA Men's tournament outcomes and save them to `data/trained_params.json` for the simulator notebook.

## Data required

Download the Kaggle March Mania dataset and place these files in `data/kaggle/`:
- `MNCAATourneyCompactResults.csv`
- `MNCAATourneySeeds.csv`
- `MTeams.csv`
- `MRegularSeasonDetailedResults.csv` (for season stats and schedule features)
- `MTeamCoaches.csv`, `MMasseyOrdinals.csv` (optional — for Kaggle-derived features)

No KenPom login required — all data comes from Kaggle files.

## Runtime
| Step | Time |
|------|------|
| Load Kaggle season stats | ~5 s |
| Build historical schedule features | ~15 s |
| Build training dataset | ~10 s |
| Train params (7 shock_df values) | ~1-2 min |
| Cross-validation | ~15-20 min |

If you want a quick pass, skip Cell 3 and train without the schedule-derived features.

In [ ]:
# Cell 1 - Imports
import os
import sys

sys.path.insert(0, ".")

from marchmadness import (
    load_kaggle_season_stats,
    build_kaggle_schedule_features,
    load_kaggle_historical,
    build_training_dataset,
    fit_feature_scaler,
    apply_feature_scaler,
    calibrate_shock_params,
    ModelParams,
    train_params,
    cross_validate,
    evaluate_params,
    predict_probs,
    save_params,
    save_scaler,
    HISTORICAL_SEED_WIN_RATES,
    build_kaggle_features_historical,
)

In [ ]:
# Cell 2 - Load historical men's data from Kaggle
#
# No browser/login required. All data comes from Kaggle CSV files.
# Stats are loaded from MRegularSeasonDetailedResults.csv.

TRAIN_YEARS = list(range(2010, 2026))

kenpom_stats = load_kaggle_season_stats(TRAIN_YEARS, "data/kaggle", gender="M")
print(f"Loaded stats for {len(kenpom_stats)} seasons: {sorted(kenpom_stats.keys())}")

kaggle_df = load_kaggle_historical(
    "data/kaggle/MNCAATourneyCompactResults.csv",
    "data/kaggle/MNCAATourneySeeds.csv",
)
print(
    f"Loaded {len(kaggle_df)} Kaggle tournament games "
    f"({kaggle_df['Season'].min()}-{kaggle_df['Season'].max()})"
)

In [ ]:
# Cell 2b - Build Kaggle-derived features  [OPTIONAL]
#
# Computes per-team features from the Kaggle box score / coach / ranking files:
#   neutral_win_pct    -> win % on neutral courts during regular season
#   coach_tourney_wins -> career NCAA tournament wins for this team's coach
#
# Requires placing these files in data/kaggle/:
#   MRegularSeasonDetailedResults.csv
#   MTeamCoaches.csv
#   MNCAATourneyCompactResults.csv
#   MMasseyOrdinals.csv
#
# If the files aren't present, set kaggle_features_df = None to skip.

try:
    kaggle_features_df = build_kaggle_features_historical(
        seasons=TRAIN_YEARS,
        kaggle_dir="data/kaggle",
        verbose=True,
    )
    print(f"\nKaggle features shape: {kaggle_features_df.shape}")
    print(f"Columns: {list(kaggle_features_df.columns)}")
    print(f"Sample (2024):")
    print(kaggle_features_df.loc[2024].head(3)[["neutral_win_pct", "neutral_games", "coach_tourney_wins", "coach_tourney_apps", "consensus_rank"]])
except FileNotFoundError as e:
    print(f"Kaggle files not found — skipping. ({e})")
    kaggle_features_df = None

In [ ]:
# Cell 3 - Build historical schedule features  [OPTIONAL]
#
# Uses Kaggle regular-season results to derive:
#   recent_form        -> opponent-weighted win rate over the last 10 games
#   season_trajectory  -> trend in cumulative weighted performance over the full season
#   conference tournament wins
#
# Uses build_kaggle_schedule_features (no KenPom login required).

historical_schedule_features, historical_conf_tourney = build_kaggle_schedule_features(
    TRAIN_YEARS, "data/kaggle", gender="M", recent_window=10
)

team_seasons = sum(len(v) for v in historical_schedule_features.values())
print(
    f"Built schedule features for {team_seasons} team-seasons across "
    f"{len(historical_schedule_features)} years."
)

# To skip these features entirely, comment out the block above and uncomment:
# historical_schedule_features = None
# historical_conf_tourney = None

In [ ]:
# Cell 4 - Build training dataset
#
# Joins Kaggle NCAA outcomes with season ratings, schedule features,
# and Kaggle-derived box score features (neutral court win%, coach experience).
# Add extra name fixes if the build step warns about unresolved teams.

extra_name_map = {
    # Example: "App State": "Appalachian St."
}

training_df = build_training_dataset(
    kaggle_df,
    kenpom_stats,
    kaggle_teams_path="data/kaggle/MTeams.csv",
    name_map=extra_name_map,
    schedule_features=historical_schedule_features,
    conf_tourney_results=historical_conf_tourney,
    kaggle_features_df=kaggle_features_df,
    min_year=2010,
)

print(f"Dataset shape: {training_df.shape}")
print(f"Label balance: {training_df['label'].mean():.3f}  (expect ~0.50)")

# Check coverage of Kaggle features
if "neutral_win_pct_a" in training_df.columns:
    pct_non_default_neutral = (training_df["neutral_win_pct_a"] != 0.5).mean()
    print(f"Neutral win pct populated (non-default): {pct_non_default_neutral:.1%} of team slots")
if "coach_tourney_wins_a" in training_df.columns:
    pct_non_zero_coach = (training_df["coach_tourney_wins_a"] > 0).mean()
    print(f"Coach tourney wins populated (>0): {pct_non_zero_coach:.1%} of team-a slots")

training_df.head()
# Fit feature scaler on the full training dataset.
feature_scaler = fit_feature_scaler(training_df)
training_df_scaled = apply_feature_scaler(training_df, feature_scaler)

print("Feature scaler stats:")
for feat, stats in feature_scaler.items():
    print(f"  {feat}: mean={stats['mean']:.4f}  std={stats['std']:.4f}")

In [ ]:
# Cell 5 - Calibrate shock parameters
#
# Finds (shock_df, shock_scale) that reproduce historical seed-matchup upset rates.
# These are fixed before feature-weight training so the optimizer can't collapse
# noise to zero in pursuit of lower log-loss.

best_margin_scale, best_shock_df, best_shock_scale = calibrate_shock_params(training_df_scaled)

params = ModelParams(
    margin_scale=best_margin_scale,
    shock_df=best_shock_df,
    shock_scale=best_shock_scale,
)
print(f"Calibrated: margin_scale={params.margin_scale:.2f}  shock_df={params.shock_df:.2f}  shock_scale={params.shock_scale:.4f}")

In [ ]:
# Cell 6 - Train parameters
#
# Strategy: outer grid search over shock_df, inner L-BFGS-B over
# margin_scale, shock_scale, luck_weight, recent_form_weight,
# trajectory_weight, and seed_prior_weight.

best_params = train_params(
    training_df_scaled,
    params_init=params,
    verbose=True,
)

metrics = evaluate_params(training_df_scaled, best_params)
print("In-sample metrics:")
print(f"  log_loss:    {metrics['log_loss']:.4f}  (random baseline = 0.6931)")
print(f"  brier_score: {metrics['brier_score']:.4f}")
print(f"  accuracy:    {metrics['accuracy']:.3f}")
print(f"  better-seed baseline accuracy: {metrics['seed_baseline_accuracy']:.3f}")
print(f"  higher-AdjEM baseline accuracy: {metrics['adj_em_baseline_accuracy']:.3f}")
print(f"  mean confidence: {metrics['mean_confidence']:.3f}")
print(
    f"  high-confidence accuracy (>=70%): {metrics['high_confidence_accuracy']:.3f} "
    f"on {metrics['high_confidence_share']:.1%} of games"
)

print("Simulated upset rates vs historical:")
for (lo, hi), hist_rate in sorted(HISTORICAL_SEED_WIN_RATES.items()):
    sim = metrics['upset_rates'].get((lo, hi), float('nan'))
    print(f"  {lo:2d}v{hi:<2d}  sim={sim:.3f}  hist={1-hist_rate:.3f}")

In [ ]:
# Cell 7 - Cross-validation  [OPTIONAL]
#
# Leave-one-year-out validation.

cv_results = cross_validate(
    training_df_scaled,
    params_init=params,
    verbose=True,
)

print("CV results:")
print(cv_results[[
    "year", "n_val", "log_loss", "brier_score", "accuracy",
    "luck_weight", "recent_form_weight",
]].to_string(index=False))

In [ ]:
# Cell 7b - Calibration table
#
# Bucket predictions into 5% bins and compare mean predicted probability
# against actual win rate. A well-calibrated model lands on the diagonal.

import numpy as np

probs = predict_probs(best_params, training_df_scaled)
labels = training_df_scaled["label"].to_numpy(dtype=float)

print("Calibration table (in-sample)")
print(f"{'Bucket':>12}  {'Pred':>6}  {'Actual':>6}  {'N':>5}  {'Gap':>6}")
print("-" * 46)
for lo in np.arange(0.0, 1.0, 0.05):
    hi = lo + 0.05
    mask = (probs >= lo) & (probs < hi)
    n = mask.sum()
    if n == 0:
        continue
    bucket_pred   = probs[mask].mean()
    bucket_actual = labels[mask].mean()
    gap = bucket_pred - bucket_actual
    flag = "  <-- overconfident" if gap > 0.07 else ("  <-- underconfident" if gap < -0.07 else "")
    print(f"{lo:5.0%}\u2013{hi:5.0%}  {bucket_pred:6.1%}  {bucket_actual:6.1%}  {n:5d}  {gap:+6.1%}{flag}")

In [ ]:
# Cell 7c - Round-conditional calibration
#
# DayNum round mapping (Kaggle convention):
#   134-135 → First Four
#   136-137 → Round of 64
#   138-140 → Round of 32
#   143-144 → Sweet 16
#   145-148 → Elite 8
#   152     → Final Four
#   154     → Championship

def _day_to_round(day: int) -> str:
    if day <= 135:  return "First Four"
    if day <= 137:  return "Round of 64"
    if day <= 140:  return "Round of 32"
    if day <= 144:  return "Sweet 16"
    if day <= 148:  return "Elite 8"
    if day <= 152:  return "Final Four"
    return "Championship"

if "day_num" not in training_df_scaled.columns:
    print("day_num column not found — re-run Cell 4 to populate it.")
else:
    round_labels_col = training_df_scaled["day_num"].apply(_day_to_round)
    round_order = ["Round of 64", "Round of 32", "Sweet 16", "Elite 8", "Final Four", "Championship"]

    print("Round-conditional calibration (in-sample)")
    print(f"{'Round':<18}  {'N':>5}  {'Pred':>6}  {'Actual':>6}  {'Acc':>5}  {'Gap':>6}")
    print("-" * 58)
    for rnd in round_order:
        mask = round_labels_col == rnd
        if not mask.any():
            continue
        p = probs[mask.to_numpy()]
        y = labels[mask.to_numpy()]
        n = mask.sum()
        mean_pred   = p.mean()
        mean_actual = y.mean()
        acc = ((p >= 0.5) == y).mean()
        gap = mean_pred - mean_actual
        flag = "  **" if abs(gap) > 0.07 else ""
        print(f"{rnd:<18}  {n:5d}  {mean_pred:6.1%}  {mean_actual:6.1%}  {acc:5.1%}  {gap:+6.1%}{flag}")

In [ ]:
# Cell 7f - Temperature scaling
#
# p_cal = sigmoid(logit / tau),  tau > 1 softens, tau < 1 sharpens.
# tau is fitted by minimizing log-loss with a single scalar.

from scipy.optimize import minimize_scalar
from sklearn.isotonic import IsotonicRegression

probs_clipped = np.clip(probs, 1e-6, 1 - 1e-6)
logits = np.log(probs_clipped / (1.0 - probs_clipped))

def _tau_logloss(tau, logits, y):
    p = 1 / (1 + np.exp(-np.clip(logits / tau, -500, 500)))
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

result = minimize_scalar(_tau_logloss, bounds=(0.5, 3.0), method="bounded",
                         args=(logits, labels))
best_tau = float(result.x)
print(f"Temperature scaling: optimal tau = {best_tau:.4f}  (tau>1 softens)")

p_temp = np.clip(1 / (1 + np.exp(-logits / best_tau)), 1e-7, 1 - 1e-7)
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(probs, labels)
p_iso = np.clip(iso.predict(probs), 1e-7, 1 - 1e-7)

def _metrics(p, y):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    ll  = float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))
    bs  = float(np.mean((p - y) ** 2))
    acc = float(np.mean((p >= 0.5) == y))
    ece = sum(mask.sum() * abs(p[mask].mean() - y[mask].mean())
              for lo in np.arange(0, 1, 0.10)
              for mask in [(p >= lo) & (p < lo + 0.10)] if mask.sum() > 0) / len(p)
    return {"log_loss": ll, "brier": bs, "acc": acc, "ece": ece}

print()
print(f"{'':30}  {'LogLoss':>7}  {'Brier':>7}  {'Acc':>5}  {'ECE':>5}  80-85%  85-90%  90-95%  95-100%")
print("-" * 100)
for name, p in [("Raw model", probs), (f"Temperature (\u03c4={best_tau:.2f})", p_temp), ("Isotonic", p_iso)]:
    m = _metrics(p, labels)
    parts = []
    for lo in [0.80, 0.85, 0.90, 0.95]:
        mask = (p >= lo) & (p < lo + 0.05)
        if mask.sum() < 3: parts.append("  n/a "); continue
        gap = p[mask].mean() - labels[mask].mean()
        parts.append(f"{gap:+5.1%}{'*' if abs(gap)>0.07 else ' '}")
    print(f"  {name:<28}  {m['log_loss']:7.4f}  {m['brier']:7.4f}  {m['acc']:5.3f}  {m['ece']:5.4f}  {'  '.join(parts)}")

In [ ]:
# Cell 7g - Cross-validate temperature tau
#
# Fit tau on each held-out year independently to check stability.
# Final tau: median of per-fold estimates.

import dataclasses

years = sorted(training_df_scaled["year"].unique())
tau_estimates = []

print(f"{'Year':>6}  {'tau':>6}  {'raw_ll':>8}  {'cal_ll':>8}  {'delta_ll':>9}")
print("-" * 46)

for val_year in years:
    train_mask = training_df_scaled["year"] != val_year
    val_mask   = training_df_scaled["year"] == val_year

    df_val  = training_df_scaled[val_mask]
    df_train = training_df_scaled[train_mask]
    y_val   = df_val["label"].to_numpy(dtype=float)
    y_train = df_train["label"].to_numpy(dtype=float)

    p_train_raw = predict_probs(dataclasses.replace(best_params, temperature=1.0), df_train)
    logits_train = np.log(np.clip(p_train_raw, 1e-6, 1-1e-6) / (1 - np.clip(p_train_raw, 1e-6, 1-1e-6)))

    def _tau_ll_train(tau):
        p = np.clip(1 / (1 + np.exp(-logits_train / tau)), 1e-9, 1-1e-9)
        return -np.mean(y_train * np.log(p) + (1 - y_train) * np.log(1 - p))

    res = minimize_scalar(_tau_ll_train, bounds=(0.5, 3.0), method="bounded")
    fold_tau = float(res.x)
    tau_estimates.append(fold_tau)

    p_val_raw = predict_probs(dataclasses.replace(best_params, temperature=1.0), df_val)
    logits_val = np.log(np.clip(p_val_raw, 1e-6, 1-1e-6) / (1 - np.clip(p_val_raw, 1e-6, 1-1e-6)))
    p_val_cal = np.clip(1 / (1 + np.exp(-logits_val / fold_tau)), 1e-9, 1-1e-9)

    ll_raw = float(-np.mean(y_val * np.log(np.clip(p_val_raw,1e-9,1-1e-9)) + (1-y_val)*np.log(np.clip(1-p_val_raw,1e-9,1-1e-9))))
    ll_cal = float(-np.mean(y_val * np.log(p_val_cal) + (1-y_val)*np.log(1-p_val_cal)))
    print(f"{val_year:>6}  {fold_tau:>6.3f}  {ll_raw:>8.4f}  {ll_cal:>8.4f}  {ll_cal-ll_raw:>+9.4f}")

tau_median = float(np.median(tau_estimates))
tau_mean   = float(np.mean(tau_estimates))
tau_std    = float(np.std(tau_estimates))
print(f"\n\u03c4 estimates:  mean={tau_mean:.3f}  median={tau_median:.3f}  std={tau_std:.3f}")
print(f"Range: [{min(tau_estimates):.3f}, {max(tau_estimates):.3f}]")
print(f"\nUsing \u03c4 = {tau_median:.4f} (median across folds)")

In [ ]:
# Cell 8 - Save trained params
import os
import dataclasses

os.makedirs("data", exist_ok=True)

try:
    final_tau = tau_median
except NameError:
    final_tau = best_tau
    print(f"Warning: tau_median not found (Cell 7g skipped). Using in-sample tau={final_tau:.4f}.")

best_params = dataclasses.replace(best_params, temperature=final_tau)

save_params(best_params, "data/trained_params.json")

print("Trained params summary:")
print(f"  margin_scale           = {best_params.margin_scale:.2f}")
print(f"  shock_df               = {best_params.shock_df:.2f}")
print(f"  shock_scale            = {best_params.shock_scale:.4f}")
print(f"  luck_weight            = {best_params.luck_weight:.4f}")
print(f"  recent_form_weight     = {best_params.recent_form_weight:.4f}")
print(f"  seed_prior_weight      = {best_params.seed_prior_weight:.4f}")
print(f"  w_efg                  = {best_params.w_efg:.4f}")
print(f"  w_to                   = {best_params.w_to:.4f}")
print(f"  w_orb                  = {best_params.w_orb:.4f}")
print(f"  w_ftr                  = {best_params.w_ftr:.4f}")
print(f"  temperature (tau)      = {best_params.temperature:.4f}")

save_scaler(feature_scaler, "data/feature_scaler.json")
print("Feature scaler saved to data/feature_scaler.json")